In [ ]:
import geopandas as gpd
import pandas as pd
import re


In [ ]:
# Load the geojson data
gdf = gpd.read_file('/Users/kaelananderson/Documents/HDMA/311 Data/data/2022/311_Encampment_Reports%2C_City_of_San_Diego%2C_2022.geojson')


In [ ]:
docs = gdf.public_des.tolist()
docs[0:5]

In [ ]:
# Regular expression pattern for encampment-related keywords
encampment_pattern = re.compile(r'\b\w*(?:tent|camp|homeless)\w*\b', re.IGNORECASE)

# Filter docs to include only those related to encampments
filtered_docs = [doc for doc in docs if encampment_pattern.search(doc) and doc.strip()]

print(f"Original number of documents: {len(docs)}")
print(f"Number of encampment-related documents: {len(filtered_docs)}")

# Ensure no empty or blank records remain
cleaned_docs = [doc for doc in filtered_docs if doc.strip()]

print(f"Number of documents after cleaning: {len(cleaned_docs)}")


In [ ]:
cleaned_docs[0:5]

# General Camp Filter

In [ ]:
# Define a broader regex pattern to filter all mentions of camps
camp_pattern = re.compile(r'\b(tent|camp|encampment|shelter|site|dwelling)\b', re.IGNORECASE)

# Apply the filter to keep only camp-related descriptions
camp_filtered_docs = [doc for doc in cleaned_docs if camp_pattern.search(doc)]

# Display the number of filtered documents and a preview
print(f"Number of camp-related documents: {len(camp_filtered_docs)}")
camp_filtered_docs[:5]  # Show the first 5 filtered descriptions


In [ ]:
camp_filtered_df = pd.DataFrame({'description': camp_filtered_docs})
camp_filtered_df.head()

## Camp subcategories 
- **Active Camps**: Descriptions indicating the presence of an occupied camp (e.g., "active," "currently," "occupied").
- **Abandoned Camps**: Descriptions mentioning abandoned or leftover items (e.g., "abandoned," "vacated," "remnants").
- **Transient Camps**: Descriptions referring to temporary setups or camps that recently moved (e.g., "transient," "temporary," "moved").


In [ ]:
# Define regex patterns for camp subcategories
active_camp_pattern = re.compile(
    r'\b(?:active|currently|present|occupied|ongoing|in use|set up|inhabited|used)\b', 
    re.IGNORECASE
)

abandoned_camp_pattern = re.compile(
    r'\b(?:abandoned|leftover|vacated|remnants|unattended|deserted|empty|forgotten|discarded|neglected|left behind)\b', 
    re.IGNORECASE
)

transient_camp_pattern = re.compile(
    r'\b(?:transient|temporary|recent|short-term|moved|relocated|not permanent|on the move|tentative|impermanent)\b', 
    re.IGNORECASE
)


# Add boolean columns for each subcategory
camp_filtered_df['Active Camp'] = camp_filtered_df['description'].str.contains(active_camp_pattern, regex=True)
camp_filtered_df['Abandoned Camp'] = camp_filtered_df['description'].str.contains(abandoned_camp_pattern, regex=True)
camp_filtered_df['Transient Camp'] = camp_filtered_df['description'].str.contains(transient_camp_pattern, regex=True)

# Summary of subcategories
print("Subcategory counts:")
print(camp_filtered_df[['Active Camp', 'Abandoned Camp', 'Transient Camp']].sum())




In [ ]:
# Preview categorized data
print("Sample of categorized data:")
camp_filtered_df.head()


# General Vehicle Filter

In [ ]:
# Define regex pattern for vehicle-related descriptions
vehicle_pattern = re.compile(r'\b(?:car|truck|van|vehicle|RV|automobile|motorhome|camper|SUV|pickup)\b', re.IGNORECASE)

# Filter descriptions mentioning vehicles
vehicle_filtered_docs = [doc for doc in cleaned_docs if vehicle_pattern.search(doc)]

# Display the number of filtered documents and a preview
print(f"Number of vehicle-related documents: {len(vehicle_filtered_docs)}")
# vehicle_filtered_docs[:5]  # Show the first 5 filtered descriptions


In [ ]:
# Create a DataFrame from the filtered descriptions
vehicle_filtered_df = pd.DataFrame({'description': vehicle_filtered_docs})
vehicle_filtered_df.iloc[0]

## Vehicle Subcategory 
- **Abandoned Filter**: Marks descriptions with terms like "abandoned," "vacant," or "unattended" as True for "Abandoned Vehicle."

In [ ]:
# Define pattern for abandoned vehicles
abandoned_vehicle_pattern = re.compile(
    r'\b(?:abandoned|vacant|left|discarded|empty|unattended|forsaken|derelict|unused|neglected|deserted|left behind|not in use|idle)\b',
    re.IGNORECASE
)

# Add boolean column for abandoned vehicles
vehicle_filtered_df['Abandoned Vehicle'] = vehicle_filtered_df['description'].str.contains(abandoned_vehicle_pattern, regex=True)

# Summary of abandoned vehicle records
print("Abandoned Vehicle Counts:")
print(vehicle_filtered_df['Abandoned Vehicle'].value_counts())


In [ ]:
# Preview categorized data
print("Sample of vehicle-related data with abandoned classification:")
vehicle_filtered_df.head()

# General Individual Filter

In [ ]:
# Define regex pattern for descriptions of individual people
person_pattern = re.compile(r'\b(?:man|woman|person|individual|guy|lady|homeless person|resident)\b', re.IGNORECASE)

# Filter descriptions mentioning individual people
people_filtered_docs = [doc for doc in cleaned_docs if person_pattern.search(doc)]


print(f"Number of people-related documents: {len(people_filtered_docs)}")


In [ ]:

# Create a DataFrame from the filtered descriptions
people_filtered_df = pd.DataFrame({'description': people_filtered_docs})
people_filtered_df.head()


## New Subcategories for Individual Descriptions

To ensure fairness and inclusivity in representing individual experiences, we have created the following subcategories based on patterns identified in the descriptions:

1. **Mental Health Crisis**:
   - Captures descriptions suggesting a mental health crisis.
   - Keywords include variations like: "screaming," "crying," "distressed," "shouting," "panic attack," "mental health," "acting out," or "erratic behavior."

2. **Neutral Presence**:
   - Identifies individuals who are simply present without any notable activity or behavior.
   - Keywords include variations like: "standing," "sitting," "sleeping," "resting," "waiting," "just there," or "quiet."

3. **Daily Activities**:
   - Includes descriptions of individuals engaging in normal day-to-day activities.
   - Keywords include variations like: "walking," "eating," "talking," "moving," "getting ready," or "looking for help."

4. **Using Drugs**:
   - Captures descriptions related to drug use without reducing individuals' experiences to this single category.
   - Keywords include variations like: "using drugs," "injecting," "smoking," "shooting up drugs," or "drug paraphernalia."

5. **Other/Uncategorized**:
   - Descriptions that do not match any of the above categories are placed here to ensure all data is accounted for.

This approach avoids reducing individuals' experiences to a single dimension (e.g., drug use) and provides a more balanced and fair representation of the data.


In [ ]:
# More inclusive regex patterns for subcategories
mental_health_pattern = re.compile(
    r'\b(?:scream\w*|cry\w*|distress\w*|shout\w*|erratic\w*|mental.*health|acting.*out|panic\w*)\b', 
    re.IGNORECASE
)

neutral_presence_pattern = re.compile(
    r'\b(?:stand\w*|sit\w*|sleep\w*|lie\w* down|rest\w*|wait\w*|just.*there|present|quiet\w*)\b', 
    re.IGNORECASE
)

daily_activities_pattern = re.compile(
    r'\b(?:walk\w*|eat\w*|talk\w*|mov\w*|gett\w* ready|look\w* for.*help|engag\w*)\b', 
    re.IGNORECASE
)

using_drugs_pattern = re.compile(
    r'\b(?:drug\w*|using.*drug\w*|doing.*drug\w*|inject\w*|smok\w*|shoot\w*.*drug\w*|paraphernalia)\b', 
    re.IGNORECASE
)


In [ ]:
# Add boolean columns for each subcategory
people_filtered_df['Mental Health Crisis'] = people_filtered_df['description'].str.contains(mental_health_pattern, regex=True)
people_filtered_df['Neutral Presence'] = people_filtered_df['description'].str.contains(neutral_presence_pattern, regex=True)
people_filtered_df['Daily Activities'] = people_filtered_df['description'].str.contains(daily_activities_pattern, regex=True)
people_filtered_df['Using Drugs'] = people_filtered_df['description'].str.contains(using_drugs_pattern, regex=True)

# Add a complementary column for "Other/Uncategorized" if no subcategory matches
people_filtered_df['Other/Uncategorized'] = ~(
    people_filtered_df['Mental Health Crisis'] |
    people_filtered_df['Neutral Presence'] |
    people_filtered_df['Daily Activities'] |
    people_filtered_df['Using Drugs']
)

# Summary of subcategories
print("Counts for fair and representative subcategories:")
print(people_filtered_df[['Mental Health Crisis', 'Neutral Presence', 'Daily Activities', 'Using Drugs', 'Other/Uncategorized']].sum())

In [ ]:
# Preview the categorized data
print("Sample of categorized individual-related data:")
people_filtered_df.head()

# Hierarchical Data Export for ArcGIS Pro

This code creates a hierarchical dataset with the following structure:
1. **First Level**: Records matching the "Homeless/Encampment Filter."
2. **Second Level**: General categories (`Camp`, `Vehicle`, `Individual`).
3. **Third Level**: Subcategories:
   - **Camps**: `Active Camp`, `Abandoned Camp`, `Transient Camp`.
   - **Vehicles**: `Abandoned Vehicle`.
   - **Individuals**: `Using Drugs`.

The dataset includes the `description` and `geometry` (spatial data) columns.

- If `geometry` is present, the data is saved as **GeoJSON** for spatial visualization in ArcGIS Pro.
- If not, the data is saved as a **CSV** for tabular analysis.


In [ ]:
# Step 1: Filter gdf to include only rows with descriptions in cleaned_docs
filtered_gdf = gdf[gdf['public_des'].isin(cleaned_docs)].copy()
filtered_gdf.head()

In [ ]:
# Step 2: Use the IDs and geometries from filtered_gdf to build the hierarchical DataFrame
hierarchical_df = pd.DataFrame({
    'record_id': filtered_gdf.index,  # Use the record IDs from the original GeoDataFrame
    'date': filtered_gdf['date_reque'],
    'description': filtered_gdf['public_des'],  # Use the descriptions that match cleaned_docs
    'geometry': filtered_gdf.geometry  # Use the correct geometry values
})

hierarchical_df

In [ ]:
# Step 3: Add hierarchical columns for filters and subcategories

# Overall encampment filter
hierarchical_df['Matches Homeless/Encampment Filter'] = [bool(encampment_pattern.search(doc)) for doc in hierarchical_df['description']]

# General filters
hierarchical_df['Camp'] = [bool(encampment_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Vehicle'] = [bool(vehicle_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Individual'] = [bool(person_pattern.search(doc)) for doc in hierarchical_df['description']]

# Camp subcategories
hierarchical_df['Active Camp'] = [bool(active_camp_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Abandoned Camp'] = [bool(abandoned_camp_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Transient Camp'] = [bool(transient_camp_pattern.search(doc)) for doc in hierarchical_df['description']]

# Vehicle subcategories
hierarchical_df['Abandoned Vehicle'] = [bool(abandoned_vehicle_pattern.search(doc)) for doc in hierarchical_df['description']]

# Individual subcategories
hierarchical_df['Mental Health Crisis'] = [bool(mental_health_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Neutral Presence'] = [bool(neutral_presence_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Daily Activities'] = [bool(daily_activities_pattern.search(doc)) for doc in hierarchical_df['description']]
hierarchical_df['Using Drugs'] = [bool(using_drugs_pattern.search(doc)) for doc in hierarchical_df['description']]


In [ ]:
# Step 4: Convert hierarchical_df to a GeoDataFrame
hierarchical_gdf = gpd.GeoDataFrame(hierarchical_df, geometry=hierarchical_df['geometry'])
hierarchical_gdf

In [ ]:
hierarchical_gdf.shape

In [ ]:
camp_count = hierarchical_gdf[hierarchical_gdf['Active Camp'] == True].shape[0]
print(f"Number of rows where Camp is True: {camp_count}")

In [ ]:
camp_count = hierarchical_gdf[hierarchical_gdf['Vehicle'] == True].shape[0]
print(f"Number of rows where Camp is True: {camp_count}")

In [ ]:
camp_count = hierarchical_gdf[hierarchical_gdf['Individual'] == True].shape[0]
print(f"Number of rows where Camp is True: {camp_count}")

In [ ]:
# Save as GeoJSON for ArcGIS Pro
output_file_path = '../data/2022/hierarchical_data_2022.geojson'
hierarchical_gdf.to_file(output_file_path, driver='GeoJSON')

print(f"Hierarchical data saved to {output_file_path}")


In [ ]:
# Save as Shapefile for ArcGIS Pro
output_file_path = '../data/2022/hierarchical_data_2022.shp'
hierarchical_gdf.to_file(output_file_path, driver='ESRI Shapefile')

print(f"Hierarchical data saved to {output_file_path}")